# box-array-to-tensor-with-recipe — faded example 3: Build the parents dict keyed by original argnum

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `box-array-to-tensor-with-recipe`. Running the beacon reports progress on the `Backprop: Box array as Tensor + recipe` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Box array as Tensor + recipe` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`box-array-to-tensor-with-recipe`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "box-array-to-tensor-with-recipe"
DD_SUBTOPIC = "Backprop: Box array as Tensor + recipe"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

`parents` records which input occupied which positional slot so the reverse pass can route gradients back by argnum. Keys are the **original** 0-based indices (never renumbered); values are the MiniTensor objects themselves (identity, not copies). Only rg=True MiniTensors are included — scalars and rg=False tensors leave a gap in the index sequence by design.

## Faded exercise 3

`box_sub` wraps a subtract over a mix of MiniTensor and scalar args. The boxing and Recipe wiring are given. **Complete the construction of `parents`** so it maps each original argnum to its MiniTensor, including only those with `requires_grad` True.

**Fill in:** parents as a dict mapping each original index to its arg, for args that are MiniTensors with requires_grad True

In [ ]:
from typing import Callable
from dataclasses import dataclass

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = np.asarray(array)
        self.requires_grad = requires_grad
        self.recipe = None

@dataclass
class Recipe:
    func: Callable
    args: tuple
    kwargs: dict
    parents: dict

def subtract(a, b):
    return a - b

def box_sub(args):
    raw_args = tuple(a.array if isinstance(a, MiniTensor) else a for a in args)
    requires_grad = any(
        isinstance(a, MiniTensor) and a.requires_grad for a in args
    )
    out_raw = subtract(*raw_args)
    parents = {}
    if requires_grad:
        parents = {idx: a for idx, a in enumerate(args)
                   if isinstance(a, MiniTensor) and a.requires_grad}
    out = MiniTensor(out_raw, requires_grad=requires_grad)
    if requires_grad:
        out.recipe = Recipe(subtract, raw_args, {}, parents)
    return out


def _test():
    p = MiniTensor(np.array([5.0, 6.0]), requires_grad=True)
    q = MiniTensor(np.array([1.0, 2.0]), requires_grad=False)
    out = box_sub((q, p))
    assert np.allclose(out.array, q.array - p.array)
    assert out.requires_grad is True
    assert out.recipe is not None
    assert out.recipe.parents == {1: p}
    assert out.recipe.parents[1] is p
    assert 0 not in out.recipe.parents
    r = MiniTensor(np.array([9.0]), requires_grad=True)
    out2 = box_sub((r, 3.0))
    assert np.allclose(out2.array, r.array - 3.0)
    assert out2.recipe.parents == {0: r}


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
from typing import Callable
from dataclasses import dataclass

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = np.asarray(array)
        self.requires_grad = requires_grad
        self.recipe = None

@dataclass
class Recipe:
    func: Callable
    args: tuple
    kwargs: dict
    parents: dict

def subtract(a, b):
    return a - b

def box_sub(args):
    raw_args = tuple(a.array if isinstance(a, MiniTensor) else a for a in args)
    requires_grad = any(
        isinstance(a, MiniTensor) and a.requires_grad for a in args
    )
    out_raw = subtract(*raw_args)
    parents = {}
    if requires_grad:
        parents = {idx: a for idx, a in enumerate(args)
                   if isinstance(a, MiniTensor) and a.requires_grad}
    out = MiniTensor(out_raw, requires_grad=requires_grad)
    if requires_grad:
        out.recipe = Recipe(subtract, raw_args, {}, parents)
    return out
```
</details>